# <center> Exploratory Data Analysis - Credit-Risk </center>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# VISUALIZATION SETTINGS
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12,5)

In [ ]:
# LOADING DATA
df = pd.read_csv('../data/raw/cs-training.csv', index_col=0)

In [ ]:
# INITIAL EXPLORATION
print(f"Shape: {df.shape}")
print(f"\n--- Info ---")
print(df.info())
print(f"\n--- Description ---")
print(df.describe())

In [ ]:
# TARGET DISTRIBUITION

target_counts = df['SeriousDlqin2yrs'].value_counts()
target_pct = df['SeriousDlqin2yrs'].value_counts(normalize=True) * 100

print("--- Contagem---")
print(target_counts)
print("\n--- Proporção (%) ---")
print(target_pct.round(2))

In [ ]:
# VISUALIZING TARGET DISTRIBUTION
fig, axes = plt.subplots(1,2, figsize=(12,4))

# PLOT 1 - ABSOLUTE VALUES
axes[0].bar(["Adimplente", "Inadimplente"], target_counts.values, color=["#2ecc71", "#e74c3c"])
axes[0].set_title("Distribuição do Target - Contagem")
axes[0].set_ylabel("Número de Clientes")
for i, v in enumerate(target_counts.values):
    axes[0].text(i,v + 500, f"{v:,}", ha="center", fontweight="bold")
    
# PLOT 2 - PERCENTUAL
axes[1].pie(target_counts.values,
            labels=["Adimplente (0)", "Inadimplente (1)"],
            colors=["#2ecc71", "#e74c3c"],
            autopct="%1.1f%%",
            startangle=90)
axes[1].set_title("Distribuição do Target — Proporção")

plt.tight_layout()
plt.savefig('../reports/target_distribution.png', dpi = 500, bbox_inches='tight')
plt.show()

In [ ]:
# MISSING VALUES EVALUATION

missing = pd.DataFrame({
    "total_missing": df.isnull().sum(),
    "pct_missing": (df.isnull().sum() / len(df) * 100).round(2)
}).query("total_missing > 0").sort_values("pct_missing", ascending=False)
print(missing)

In [ ]:
# MISSING VALUES VISUALIZATION
fig, ax = plt.subplots(figsize=(8,4))
ax.barh(missing.index, missing['pct_missing'], color="#e74c3c")
ax.set_xlabel("Porcentagem de Valores Faltantes (%)")
ax.set_title("Porcentagem de Valores Faltantes por Variável")
for i, v in enumerate(missing['pct_missing']):
    ax.text(v + 0.2, i, f"{v:.2f}%", va="center", fontweight="bold")
plt.tight_layout()
plt.savefig('../reports/missing_values.png', dpi = 150, bbox_inches='tight')
plt.show()

In [ ]:
# EVALUTING THE MISSING VALUES RANDOMNESS
# comparing monthly income with serious delinquency

tem_renda = df[df['MonthlyIncome'].notna()]["SeriousDlqin2yrs"].mean()
sem_renda = df[df['MonthlyIncome'].isna()]["SeriousDlqin2yrs"].mean()

print(f"\n Taxa de inadimplência para clientes com renda mensal: {tem_renda:.2f}")
print(f"\n Taxa de inadimplência para clientes sem renda mensal: {sem_renda:.2f}")
print(f"\nDiferença: {abs(tem_renda - sem_renda):.2%}")

In [ ]:
# CREATING A BINARY FEATURE FOR MISSING INCOME
df["flag_missing_income"] = df["MonthlyIncome"].isna().astype(int)
df["flag_missing_dependents"] = df["NumberOfDependents"].isna().astype(int)

In [ ]:
# IMPUTATION WITH MEDIAN FOR MONTHLY INCOME AND NUMBER OF DEPENDENTS
df['MonthlyIncome'] = df['MonthlyIncome'].fillna(df['MonthlyIncome'].median())
df['NumberOfDependents'] = df['NumberOfDependents'].fillna(df['NumberOfDependents'].median())

In [ ]:
# CHECKING THE DATA AFTER IMPUTATION
print(df.isnull().sum().sum(), "nulos restantes")

In [ ]:
print(df[['flag_missing_income', 'flag_missing_dependents']].value_counts())

In [ ]:
# FEATURE DISTRIBUTIONS AND OUTLIER ANALYSIS
features_numericas = ["RevolvingUtilizationOfUnsecuredLines", "age", "DebtRatio",
    "MonthlyIncome", "NumberOfOpenCreditLinesAndLoans",
    "NumberRealEstateLoansOrLines", "NumberOfDependents"]

In [ ]:
fig, axes = plt.subplots(3,3, figsize=(15,12))
axes = axes.flatten()

for i, col in enumerate(features_numericas):
    axes[i].hist(df[col], bins=50, color="#3498db", edgecolor="white", alpha=0.8)
    axes[i].set_title(col, fontsize=10)
    axes[i].set_xlabel("Valor")
    axes[i].set_ylabel("Frequência")
    
# HIDING EMPTY SUBPLOT
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)
    
plt.suptitle("Distribuição das Variáveis Numéricas", fontsize=14, fontweight="bold", y= 1.01)
plt.tight_layout()
plt.savefig('../reports/numeric_distributions.png', dpi = 150, bbox_inches='tight')
plt.show()

In [ ]:
# OUTLIERS INVESTIGATION

print("=== RevolvingUtilization ===")
print(f"% acima de 1.0: {(df['RevolvingUtilizationOfUnsecuredLines'] > 1).mean():.2%}")
print(f"Máximo: {df['RevolvingUtilizationOfUnsecuredLines'].max():.2f}")

print("\n=== Age ===")
print(f"Clientes com age == 0: {(df['age'] == 0).sum()}")
print(f"Clientes com age > 90: {(df['age'] > 90).sum()}")

print("\n=== DebtRatio ===")
print(f"% acima de 1.0: {(df['DebtRatio'] > 1).mean():.2%}")
print(f"% acima de 100: {(df['DebtRatio'] > 100).mean():.2%}")

print("\n=== MonthlyIncome ===")
print(f"Máximo: {df['MonthlyIncome'].max():,.0f}")
print(f"Percentil 99%: {df['MonthlyIncome'].quantile(0.99):,.0f}")
print(f"Percentil 95%: {df['MonthlyIncome'].quantile(0.95):,.0f}")

In [ ]:
# TREATING OUTLIERS

# REMOVING REGISTERS WITH AGE == 0
print(f"Registros antes: {len(df)}")
df = df[df['age'] != 0].copy()
print(f"Registros depois: {len(df)}")

In [ ]:
# INVESTIGATING AGE > 90 BEFORE DECIDING TO REMOVE
print("\n--- Clientes com age > 90 ---")
print(df[df['age']>90]['age'].value_counts().sort_index())
print(f"Taxa inadimplencia age > 90: {df[df['age']>90]['SeriousDlqin2yrs'].mean():.2%}")
print(f"Taxa inadimplencia geral: {df['SeriousDlqin2yrs'].mean():.2%}")

In [ ]:
# WINSORING

colunas_winsor = {"RevolvingUtilizationOfUnsecuredLines": 0.99, "DebtRatio": 0.99, "MonthlyIncome": 0.99}

for col, percentil in colunas_winsor.items():
    p = df[col].quantile(percentil)
    antes = df[col].max()
    df[col] = df[col].clip(upper=p)
    print(f"{col}: max {antes:,.0f} -> {df[col].max():,.0f}")

In [ ]:
# FINAL CHECK
print('\n--- Estatisticas após tratamento de outliers ---')
print(df[list(colunas_winsor.keys())].describe().round(2))

In [ ]:
# TREATING DEBT RATIO OUTLIERS WITH P95

print("Percentis de DebtRatio:")
for p in [0.90, 0.95, 0.97, 0.99]:
    print(f" p{int(p*100)}: {df['DebtRatio'].quantile(p):.2f}")


In [ ]:
# WINSORING DEBT RATIO AT P95
p95_debt = df['DebtRatio'].quantile(0.95)
df['DebtRatio'] = df['DebtRatio'].clip(upper=p95_debt)
print(f"\nDebtRatio apos p95: max = {df['DebtRatio'].max():,.2f}")
print(f"Media apos correcao: {df['DebtRatio'].mean():.2f}")

In [ ]:
# INVESTIGATION OF DEBT RATIO FURTHER
print("Distribuição de DebtRatio por faixas:")
faixas = [0, 0.5, 1, 2, 5, 10, 100, 1000, 10000]
print(pd.cut(df["DebtRatio"], bins=faixas).value_counts().sort_index())

In [ ]:
# EVALUATING BINS
plausivel = (df["DebtRatio"] < 2).mean()
print(f"\n% com DebtRatio < 2 (plausível): {plausivel:.2%}")
print(f"% com DebtRatio >= 2 (suspeito): {(1-plausivel):.2%}")

In [ ]:
# INADIMPLÊNCIA POR FAIXAS DE DEBT RATIO
df["debt_faixa"] = pd.cut(df["DebtRatio"], bins=[0, 1, 2, 10, 100, 99999],
                           labels=["0-1", "1-2", "2-10", "10-100", "100+"])
print("\nTaxa inadimplência por faixa de DebtRatio:")
print(df.groupby("debt_faixa", observed=True)["SeriousDlqin2yrs"].agg(["mean", "count"]).round(3))

In [ ]:
# CLEANING UP TEMPORARY COLUMNS
df = df.drop(columns=["debt_faixa"])

In [ ]:
# TESTING THE HIPOTHESIS OF MONTHLY INCOME = 0 OR MISSING BEING SIMILAR TO HIGH DEBT RATIO
print("=== Investigando origem do DebtRatio alto ===")

# Cruzar DebtRatio alto com flag de missing income
alto = df[df["DebtRatio"] > 100]
baixo = df[df["DebtRatio"] <= 100]

print(f"DebtRatio > 100 — % com flag_missing_income=1: {alto['flag_missing_income'].mean():.2%}")
print(f"DebtRatio <= 100 — % com flag_missing_income=1: {baixo['flag_missing_income'].mean():.2%}")

# Distribuição de MonthlyIncome para quem tem DebtRatio > 100
print(f"\nDebtRatio > 100 — MonthlyIncome médio: {alto['MonthlyIncome'].mean():,.0f}")
print(f"DebtRatio <= 100 — MonthlyIncome médio: {baixo['MonthlyIncome'].mean():,.0f}")

In [ ]:
# ── CRITICAL EDA OBSERVATION ───────────────────────────────────
# 93% dos registros com DebtRatio > 100 correspondem a clientes sem renda informada (flag_missing_income = 1).
#
# Hipótese confirmada: DebtRatio foi calculado na origem como dívida / MonthlyIncome. Quando MonthlyIncome era nulo (zero ou ausente), o resultado gerou valores absurdos.
#
# Consequência: DebtRatio não é confiável para ~16k registros.
# Estratégia: winsorizar pelo p99 do subconjunto plausível (< 2),
# isolando o limite real do dado corrompido.
# ────────────────────────────────────────────────────────────

In [ ]:
# FINAL DECISION — WINSORING ON p99 
# Usamos apenas quem tem DebtRatio < 2 para calcular o limite real
p99_plausivel = df[df["DebtRatio"] < 2]["DebtRatio"].quantile(0.99)
print(f"\nP99 do grupo plausível (DebtRatio < 2): {p99_plausivel:.4f}")

df["DebtRatio"] = df["DebtRatio"].clip(upper=p99_plausivel)
print(f"DebtRatio após winsorização contextual: max = {df['DebtRatio'].max():.4f}")
print(f"Média após correção: {df['DebtRatio'].mean():.4f}")

In [ ]:
#"O DebtRatio está corrompido para clientes sem renda informada. 
# Mas já criamos flag_missing_income = 1 para identificar exatamente esse grupo. 
# O modelo pode aprender que quando flag_missing_income = 1, o DebtRatio não é confiável
# e usar as outras features para tomar a decisão."

In [ ]:
# FINAL VERIFICATION OF THE DATASET
print("--- Estado atual do dataset ---")
print(f"Registros: {len(df):,}")
print(f"Features: {df.shape[1]}")
print(f"Nulos restantes: {df.isnull().sum().sum()}")
print(f"\nFeatures Disponiveis:")
print(df.columns.tolist())

In [ ]:
# TARGET CORRELATIONS
correlacoes = df.corr()['SeriousDlqin2yrs'].drop('SeriousDlqin2yrs').sort_values(ascending=False)
print("--- Correlações com o Target ---")
print(correlacoes.round(4))

In [ ]:
# VISUALIZING CORRELATIONS
fig, ax = plt.subplots(figsize=(10,6))
cores = ["#e74c3c" if v > 0 else "#2ecc71" for v in correlacoes.values]
ax.barh(correlacoes.index, correlacoes.values, color=cores)
ax.set_title("Correlacoes das Variáveis com o Target (SeriousDlqin2yrs)", fontsize=14, fontweight="bold")
ax.set_xlabel("Correlação de Pearson")
plt.tight_layout()
plt.savefig('../reports/target_correlations.png', dpi = 150, bbox_inches='tight')
plt.show()

In [ ]:
# CORRELATION MATRIX
fig, ax = plt.subplots(figsize=(13,9))
mask = np.triu(np.ones_like(df.corr(), dtype=bool))
sns.heatmap(df.corr().round(2),
            mask = mask,
            annot=True,
            fmt=".2f",
            cmap="RdYlGn_r",
            center=0,
            ax=ax,
            annot_kws={"size":8})
ax.set_title("Matriz de Correlação", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig('../reports/correlation_matrix.png', dpi = 150, bbox_inches='tight')
plt.show()

In [ ]:
# DELINQUENCY RATE BY AGE GROUPS
df["age_faixa"] = pd.cut(df["age"],
                          bins=[0, 25, 35, 45, 55, 65, 110],
                          labels=["<25", "25-35", "35-45", "45-55", "55-65", "65+"])

print("\n=== Taxa de inadimplência por faixa etária ===")
print(df.groupby("age_faixa", observed=True)["SeriousDlqin2yrs"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "taxa_inadimplencia", "count": "clientes"})
        .round(3))

df = df.drop(columns=["age_faixa"])

In [ ]:
# INFORMATION VALUE FUNCTION
def calcular_iv(df, feature, target, bins=10):
    df_temp = df[[feature, target]].copy()
    
    # Discretizar features contínuas
    if df_temp[feature].nunique() > 10:
        df_temp["bin"] = pd.qcut(df_temp[feature], q=bins, duplicates="drop")
    else:
        df_temp["bin"] = df_temp[feature]
    
    # Calcular WoE e IV por bin
    total_events = df_temp[target].sum()
    total_non_events = len(df_temp) - total_events
    
    iv_table = df_temp.groupby("bin", observed=True)[target].agg(
        events="sum",
        total="count"
    )
    iv_table["non_events"] = iv_table["total"] - iv_table["events"]
    iv_table["pct_events"] = iv_table["events"] / total_events
    iv_table["pct_non_events"] = iv_table["non_events"] / total_non_events
    
    # Evitar log(0)
    iv_table = iv_table[(iv_table["pct_events"] > 0) & (iv_table["pct_non_events"] > 0)]
    
    iv_table["woe"] = np.log(iv_table["pct_events"] / iv_table["pct_non_events"])
    iv_table["iv"] = (iv_table["pct_events"] - iv_table["pct_non_events"]) * iv_table["woe"]
    
    return iv_table["iv"].sum()


In [ ]:
# CALCULATING IV FOR ALL FEATURES
features = [c for c in df.columns if c != "SeriousDlqin2yrs"]
iv_scores = {f: calcular_iv(df, f, "SeriousDlqin2yrs") for f in features}
iv_df = pd.DataFrame.from_dict(iv_scores, orient="index", columns=["IV"])\
          .sort_values("IV", ascending=False)

print("=== Information Value por Feature ===")
print(iv_df.round(4))

In [ ]:
# IV CLASSIFICATION FUNCTION
def classificar_iv(iv):
    if iv < 0.02: return "Fraco"
    elif iv < 0.1: return "Médio"
    elif iv < 0.3: return "Forte"
    else: return "Suspeito (possível leakage)"


In [ ]:
# CLASSIFYING FEATURES BY IV
iv_df["classificacao"] = iv_df["IV"].apply(classificar_iv)
print("\n=== Classificação ===")
print(iv_df)

In [ ]:
# INVESTIGATING REVOLVING UTILIZATION IV
print("=== RevolvingUtilization por faixa ===")
df["revolving_faixa"] = pd.qcut(
    df["RevolvingUtilizationOfUnsecuredLines"], q=10, duplicates="drop"
)
print(df.groupby("revolving_faixa", observed=True)["SeriousDlqin2yrs"]
        .agg(["mean", "count"])
        .rename(columns={"mean": "taxa_inadimplencia", "count": "clientes"})
        .round(3))
df = df.drop(columns=["revolving_faixa"])

In [ ]:
# INVESTIGATING FEATURES WITH IV = 0
print("\n=== Distribuição NumberOfTimes90DaysLate ===")
print(df["NumberOfTimes90DaysLate"].value_counts().sort_index().head(10))
print(f"% com valor 0: {(df['NumberOfTimes90DaysLate'] == 0).mean():.2%}")

print("\n=== Distribuição NumberOfTime60-89DaysPastDueNotWorse ===")
print(df["NumberOfTime60-89DaysPastDueNotWorse"].value_counts().sort_index().head(10))
print(f"% com valor 0: {(df['NumberOfTime60-89DaysPastDueNotWorse'] == 0).mean():.2%}")

In [ ]:
# RECALCULATING IV AFTER INVESTIGATION (LESS BINS)
for col in ["NumberOfTimes90DaysLate", "NumberOfTime60-89DaysPastDueNotWorse",
            "NumberOfTime30-59DaysPastDueNotWorse"]:
    iv = calcular_iv(df, col, "SeriousDlqin2yrs", bins=5)
    print(f"\nIV recalculado ({col}): {iv:.4f}")

In [ ]:
#A relação é monotônica e gradual — quanto maior a utilização do crédito rotativo, maior a inadimplência. 
# Sem saltos abruptos, sem comportamento anômalo.
# Conclusão: não é leakage. O IV alto (1.11) reflete que essa feature é genuinamente muito preditiva 
# e faz sentido de negócio. 
# Quem usa 98% do limite do cartão tem risco real de inadimplir. 
# O IV > 0.3 como "suspeito" é uma heurística conservadora, não uma regra absoluta.

In [ ]:
# RECALCULATING IV FOR FEATURES WITH EXTREME DISTRIBUITIONS
def iv_contagem(df, col, target):
    """IV específico para variáveis de contagem com alta concentração em 0"""
    df_temp = df[[col, target]].copy()
    
    # Bins de negócio: 0, 1, 2, 3+
    df_temp["bin"] = pd.cut(
        df_temp[col],
        bins=[-1, 0, 1, 2, 999],
        labels=["0", "1", "2", "3+"]
    )
    
    total_events = df_temp[target].sum()
    total_non_events = len(df_temp) - total_events
    
    iv_table = df_temp.groupby("bin", observed=True)[target].agg(
        events="sum", total="count"
    )
    iv_table["non_events"] = iv_table["total"] - iv_table["events"]
    iv_table["pct_events"] = iv_table["events"] / total_events
    iv_table["pct_non_events"] = iv_table["non_events"] / total_non_events
    iv_table = iv_table[
        (iv_table["pct_events"] > 0) & (iv_table["pct_non_events"] > 0)
    ]
    iv_table["woe"] = np.log(
        iv_table["pct_events"] / iv_table["pct_non_events"]
    )
    iv_table["iv"] = (
        (iv_table["pct_events"] - iv_table["pct_non_events"]) * iv_table["woe"]
    )
    
    print(f"\n=== {col} ===")
    print(iv_table[["events", "non_events", "woe", "iv"]].round(4))
    print(f"IV Total: {iv_table['iv'].sum():.4f}")
    
    return iv_table["iv"].sum()

In [ ]:
cols_contagem = [
    "NumberOfTimes90DaysLate",
    "NumberOfTime60-89DaysPastDueNotWorse",
    "NumberOfTime30-59DaysPastDueNotWorse"
]

ivs_corrigidos = {}
for col in cols_contagem:
    ivs_corrigidos[col] = iv_contagem(df, col, "SeriousDlqin2yrs")

In [ ]:
# CONSOLIDATING IV RESULTS
iv_final = {
    "RevolvingUtilizationOfUnsecuredLines": 1.1134,
    "NumberOfTimes90DaysLate": 0.8774,
    "NumberOfTime30-59DaysPastDueNotWorse": 0.7552,
    "NumberOfTime60-89DaysPastDueNotWorse": 0.6000,
    "age": 0.2592,
    "MonthlyIncome": 0.0670,
    "NumberOfOpenCreditLinesAndLoans": 0.0669,
    "DebtRatio": 0.0253,
    "NumberOfDependents": 0.0250,
    "NumberRealEstateLoansOrLines": 0.0121,
    "flag_missing_income": 0.0077,
    "flag_missing_dependents": 0.0037
}

iv_final_df = pd.DataFrame.from_dict(
    iv_final, orient="index", columns=["IV"]
).sort_values("IV", ascending=True)

# Visualização
fig, ax = plt.subplots(figsize=(10, 7))
cores = ["#e74c3c" if v > 0.3 else "#f39c12" if v > 0.1
         else "#2ecc71" if v > 0.02 else "#95a5a6"
         for v in iv_final_df["IV"]]
ax.barh(iv_final_df.index, iv_final_df["IV"], color=cores)
ax.axvline(0.02, color="gray", linestyle="--", alpha=0.7, label="Fraco (0.02)")
ax.axvline(0.1, color="#f39c12", linestyle="--", alpha=0.7, label="Médio (0.1)")
ax.axvline(0.3, color="#e74c3c", linestyle="--", alpha=0.7, label="Forte (0.3)")
ax.set_title("Information Value — Ranking de Poder Preditivo", fontweight="bold")
ax.set_xlabel("IV")
ax.legend()
plt.tight_layout()
plt.savefig("../reports/iv_ranking.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# SAVING THE DATASET CLEANED
df.to_csv("../data/processed/credit_processed.csv", index=False)
print(f"Dataset salvo: {df.shape}")

In [ ]:
# EDA RESUME
decisoes = {
    "registros_originais": 150000,
    "registros_finais": len(df),
    "registros_removidos": 150000 - len(df),
    "features_originais": 11,
    "features_criadas": ["flag_missing_income", "flag_missing_dependents"],
    "tratamentos_aplicados": [
        "Removido 1 registro com age=0 (dado inválido)",
        "Winsorização RevolvingUtilization no p99",
        "Winsorização MonthlyIncome no p99",
        "Winsorização DebtRatio no p99 do grupo plausível (DebtRatio < 2)",
        "Imputação MonthlyIncome com mediana (19.8% missing)",
        "Imputação NumberOfDependents com mediana (2.6% missing)",
        "Criação de flags de missing antes da imputação"
    ],
    "achados_criticos": [
        "DebtRatio corrompido para 93% dos clientes sem renda informada",
        "Features de atraso com 94%+ de zeros — IV calculado com bins de negócio",
        "RevolvingUtilization com IV=1.11 investigado e confirmado como legítimo",
        "Relação monotônica entre idade e inadimplência confirmada",
        "Um único atraso de 90 dias já eleva drasticamente o risco"
    ]
}

import json
with open("../reports/eda_decisions.json", "w", encoding="utf-8") as f:
    json.dump(decisoes, f, ensure_ascii=False, indent=2)

print("\n=== RESUMO FINAL DA EDA ===")
for k, v in decisoes.items():
    print(f"\n{k}:")
    if isinstance(v, list):
        for item in v:
            print(f"  • {item}")
    else:
        print(f"  {v}")